# Day 82 — Autograd

This notebook coverse PyTorch Autograd & Dynamic Computation Graphs (`DAG`), Leaf vs. Non-Leaf Tensors, Analytical Derivative Verification via `.backward()`, Gradient Accumulation & Zeroing (`.zero_()`), and Inference Context Management (`torch.no_grad()`)

## Validation goals
- Understand how PyTorch builds a Directed Acyclic Graph (`DAG`) dynamically when `requires_grad=True` is set on leaf tensors
- Compute exact analytical derivatives via `.backward()` and verify scalar output requirements
- Diagnose gradient accumulation across consecutive `.backward()` calls and reset gradients properly using `.zero_()`
- IDisable computation graph tracking during validation and inference using `torch.no_grad()` to reduce memory overhead and improve speed

### 1. How Autograd and the Computation Graph Work

When a tensor has `requires_grad=True`, PyTorch tracks every operation performed on it to build a Directed Acyclic Graph (`DAG`) on the fly:
- Leaf Tensors: Tensors created directly by you (such as model weights, biases, or a parameter tensor with `requires_grad=True`) that are not the result of an operation. Their gradients accumulate in `.grad`.
- Non-Leaf Intermediate Tensors: Outputs of intermediate operations (like `z = w * x + b`). By default, PyTorch frees their intermediate gradient buffers after backward passes to save memory.
- `.backward()`: Starting from a scalar output tensor (such as a loss value), `.backward()` walks backward through the graph using the chain rule and populates `.grad` on leaf tensors with `requires_grad=True`.

### 2. Gradients in Action: A Concrete Example

Let's trace a small differentiable expression where we can verify the math by hand:$$y = 3w^2 + 2w + 5$$Taking the derivative with respect to $w$:$$\frac{dy}{dw} = 6w + 2$$At $w = 4.0$, the exact derivative is $6(4.0) + 2 = 26.0$. Let's watch PyTorch compute this automatically:

In [1]:
import torch

# Create a leaf parameter tensor with gradient tracking enabled
w = torch.tensor(4.0, dtype=torch.float32, requires_grad=True)

# Forward computation builds the autograd graph dynamically
y = 3.0 * (w ** 2) + 2.0 * w + 5.0

# Compute gradients of scalar y with respect to w
y.backward()

print("--- Day 82: Autograd Derivative Check ---")
print(f"w value:      {w.item():.2f}")
print(f"y value:      {y.item():.2f}")
print(f"dy/dw (.grad):{w.grad.item():.2f}")
print(f"Is leaf?      {w.is_leaf}")

--- Day 82: Autograd Derivative Check ---
w value:      4.00
y value:      61.00
dy/dw (.grad):26.00
Is leaf?      True


### 3. Gradient Accumulation and Zeroing Gradients

A critical rule in PyTorch autograd: `.grad` accumulates (adds) across calls rather than overwriting.

Why? In architectures like Recurrent Neural Networks or gradient accumulation across micro-batches, summing gradients across steps is useful. In standard mini-batch training loops, however, forgetting to clear old gradients causes stale gradients from step $t-1$ to leak into step $t$.

In [2]:
import torch

x = torch.tensor(2.0, dtype=torch.float32)
w = torch.tensor(3.0, dtype=torch.float32, requires_grad=True)

# Step 1: loss = (w * x - 10)^2 -> d(loss)/dw = 2 * (6 - 10) * 2 = -16
loss1 = (w * x - 10.0) ** 2
loss1.backward()
print(f"After 1st backward, w.grad = {w.grad.item():.1f}")

# Step 2 WITHOUT zeroing: w.grad ADDS the new gradient (-16) to the existing (-16)
loss2 = (w * x - 10.0) ** 2
loss2.backward()
print(f"After 2nd backward (un-zeroed), w.grad = {w.grad.item():.1f}")

# Reset gradient before computing a fresh gradient
w.grad.zero_()
loss3 = (w * x - 10.0) ** 2
loss3.backward()
print(f"After zeroing + 3rd backward, w.grad = {w.grad.item():.1f}")

After 1st backward, w.grad = -16.0
After 2nd backward (un-zeroed), w.grad = -32.0
After zeroing + 3rd backward, w.grad = -16.0


### 4. Disabling Gradient Tracking: `torch.no_grad()`

Building and storing a computation graph costs memory and compute. During inference, validation, or test evaluation, you are not calling `.backward()`, so you do not need graph tracking.

Wrapping evaluation or inference blocks in with `torch.no_grad()`: disables gradient recording inside that block, cutting memory overhead and speeding up execution:

In [3]:
import torch

x = torch.tensor([[1.0, 2.0, 3.0]], dtype=torch.float32)
w = torch.tensor([[0.5], [-0.2], [0.8]], dtype=torch.float32, requires_grad=True)

with torch.no_grad():
    pred = x @ w
    print(f"Inference output: {pred.item():.2f}, requires_grad: {pred.requires_grad}")

Inference output: 2.50, requires_grad: False


### Guided Question:

Why does `.backward()` require the starting tensor (such as loss) to be a scalar in standard usage, and how does `autograd` apply the chain rule across operations?

### Answer:

**1. Why `.backward()` Expects a Scalar by Default**

In training, we usually want to know how one single final score—your `loss—changes` when you nudge each weight or input slightly. That single summary number is a scalar (a 0D tensor with just one value).

Imagine you run a shop and track your total monthly profit (loss or error score). You ask: "If I adjust the price of rice by ₦10, how does my total profit change?" Because total profit is a single number, there is a clear, unambiguous rate of change for every knob you turn.

Now suppose instead you had a vector of 100 separate numbers and just called `tensor.backward()` with no extra info. Which of those 100 numbers is PyTorch supposed to measure sensitivity for? A vector output relative to a vector input has a whole Jacobian matrix of partial derivatives—there isn't one single scalar "goal" to differentiate by default.
- Scalar output (`loss.backward()`): PyTorch assumes an implicit starting gradient of 1.0 (meaning "how much does loss change with respect to loss itself?"—which is $1$).Non-scalar output (`v.backward(grad_tensor)`): If you do call `.backward()` on a non-scalar tensor, PyTorch requires you to pass a matching gradient tensor (representing external upstream weights/sensitivities) so it can compute a vector-Jacobian product and collapse it into a single direction of change per parameter.

**2. How Autograd Applies the Chain Rule Across Operations**

Think of a forward pass as an assembly line where raw materials (x) pass through stations (`nn.Linear`, `ReLU`, subtraction, squaring) until they become a final quality score (`loss`).

As data moves forward, PyTorch records a directed acyclic graph (`DAG`) of operations (`grad_fn`): each node remembers which operation ran and saves whatever intermediate values are needed to compute its local derivative later.

The Chain Rule in Plain TermsSuppose changing Station A changes Station B by a factor of 3, and changing Station B changes the final Profit/Loss by a factor of 2. How much does changing Station A affect the final Profit/Loss?
- You multiply the rates along the path: $2 \times 3 = 6$.
- In calculus for a chain of functions $x \to u \to v \to L$:$$\frac{dL}{dx} = \frac{dL}{dv} \cdot \frac{dv}{du} \cdot \frac{du}{dx}$$

**How Autograd Executes This Backward**
- Start at the end (loss): Autograd sets the incoming ("upstream") gradient at the final output to 1.0.Step backward one operation at a time: At each operation node, autograd takes the upstream gradient already arriving from later layers and multiplies it by that operation's local derivative (how much this specific step's output changes with respect to its own inputs).
- Accumulate gradients at leaves: When the backward pass reaches leaf tensors with requires_grad=True (like your model weights w and biases b), it adds (+=) the resulting gradient into tensor.grad. (That's also why we call optimizer.zero_grad() each step so old gradients don't stack up unintentionally.)
- Branching paths sum up: If a tensor feeds into two different branches later in the network, the chain rule says its total gradient is the sum of the gradients coming back from each branch:$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y_1}\frac{\partial y_1}{\partial x} + \frac{\partial L}{\partial y_2}\frac{\partial y_2}{\partial x}$$

Autograd naturally sums those incoming branches as it walks the graph in reverse.

**Everyday Analogy: Adjusting a Recipe**

Imagine baking bread where your final score (loss) is taste rating:
- Oven temperature (x) affects crust browning (u).
- Crust browning (u) affects aroma & crunch (v).
- Aroma & crunch (v) affects taste rating (L).

To figure out how turning the oven dial up by 1 degree affects the final taste rating, you don't re-bake every combination from scratch. You work backward from the taste score: multiply "how taste responds to crunch/aroma" $\times$ "how crunch/aroma responds to browning" $\times$ "how browning responds to oven temperature." That backward pass of multiplying local sensitivities is autograd applying the chain rule.

### Guided Question:

What happens if you call `loss.backward()` on consecutive training iterations without clearing previous gradients, and why is that a bug in standard training loops?

### Answer:

If you call `loss.backward()` across consecutive training steps without clearing old gradients (for example, via `optimizer.zero_grad()`), PyTorch adds (+=) the new gradients on top of the existing tensor.grad values instead of replacing them.

In a standard mini-batch training loop, leaving old gradients in place is a bug because each batch is supposed to estimate the gradient for that current batch, not accumulate history across unrelated batches.

**Everyday Analogy: Tracking Daily Expenses**

Imagine a shop owner who wants to know how much money came in today so they can decide how much restock cash to set aside.
- With `optimizer.zero_grad()` (clearing): At the start of each day, you wipe the daily tally board clean, count today's sales, and make your decision based on today's numbers.
- Without `optimizer.zero_grad()` (forgetting to clear): You leave Monday's sales on the board, add Tuesday's sales on top, add Wednesday's on top of that, and so on. By Friday, the board says you made five times more in a single day than you actually did—so you make oversized decisions based on stale, stacked numbers.

Gradients in PyTorch work just like that tally board: `loss.backward()` computes the slope for the current batch and adds it to whatever is already sitting in `.grad`.

**Why This Breaks Standard Training Loops**

- Effective gradient magnitude keeps growing: On step 1, grad equals $g_1$. On step 2 (if not cleared), grad becomes $g_1 + g_2$. On step 3, $g_1 + g_2 + g_3$, and so on.
- Oversized weight updates and instability: When optimizer.step() runs, it updates parameters using the accumulated .grad. Larger-than-intended gradients can cause big jumps in weights, oscillating loss, or exploding/unstable training—especially when combined with momentum-based optimizers like SGD with momentum or Adam.
- Wrong direction when batches disagree: Suppose Batch A pushes a weight slightly up ($+0.4$) and Batch B pushes that same weight slightly down ($-0.3$). If you don't zero gradients, Batch B's step still carries $+0.4 - 0.3 = +0.1$ of leftover pull from Batch A instead of reflecting Batch B alone.

### Guided Question:

Why should validation and test loops run inside with `torch.no_grad()`: instead of tracking gradients like during training?

### Answer:

Running validation and test loops inside with `torch.no_grad()`: tells PyTorch not to build the autograd computation graph or track gradients during evaluation. Because validation and test sets are only used to measure performance—never to update model weights via .backward() and optimizer.step()—recording gradient history there is wasted work.

Everyday Analogy: Checking Route Progress vs. Rebuilding the Engine
Imagine driving from Lagos to Ibadan:
- Training is like tuning the engine while driving: You constantly test adjustments, note how each tweak affects speed (loss), and keep notes on how every part connects so you can wrench on the engine (optimizer.step()) to go faster.
- Validation/testing is like checking a map and odometer at a toll gate: You just want to see how far you've come and how much fuel you used. You don't need to keep mechanical notes for adjusting the engine—you're just reading the meter.

**Why with `torch.no_grad()`: Matters in Validation & Test Loops**
- Lower memory usage (fewer out-of-memory crashes): During training, PyTorch saves intermediate tensors and operation history (grad_fn) needed for the backward pass. Turning that off frees up significant RAM/VRAM, allowing larger validation batch sizes or bigger models without running out of memory.
- Faster evaluation speed: Skipping graph construction and gradient bookkeeping avoids extra Python/C++ overhead on every forward pass, so validation and test checks finish noticeably faster.
- Cleaner, safer evaluation: You never accidentally leave a gradient graph hanging around or trigger `.backward()` on validation loss by mistake.

**Quick Reminder: `no_grad()` vs. `model.eval()`**

They solve two different things and are typically used together during validation/testing:
- `model.eval()` switches layer behavior to evaluation mode—for example, Dropout stops randomly dropping neurons, and BatchNorm uses running batch statistics instead of updating them with the current batch.
- with `torch.no_grad()`: disables autograd tracking so memory isn't spent storing graph history for backpropagation.